# RF-3DGS — radio radiance field reconstruction

**Tanisha Jain · Experiment notebook**

Load the visual initialization checkpoint, train on RF spectrum images, and compare reconstructed RF views.


Run this notebook in a **fresh Google Colab GPU runtime**. These are curated experiment
workflows, based on my original testing notebooks. Upstream implementations do the model
training and rendering. CUDA/PyTorch compatibility must be checked against the upstream
environment before installing extensions. The cleaned notebook has been syntax-checked,
but has not been rerun end-to-end on a GPU for this publication.

Implementation and data instructions: [SunLab-UGA/RF-3DGS](https://github.com/SunLab-UGA/RF-3DGS).

## 1. Check the runtime and clone the implementation

In [ ]:
!nvidia-smi
import torch
from pathlib import Path
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Select a GPU runtime."
%cd /content
!git clone --recursive https://github.com/SunLab-UGA/RF-3DGS.git
%cd /content/RF-3DGS
!git rev-parse HEAD

## 2. Build dependencies
This retains the `<cfloat>` compatibility fix used during the original experiment. If a build fails, align PyTorch, the CUDA toolkit, and compiler versions with the upstream setup instructions.

In [ ]:
source = Path("submodules/simple-knn/simple_knn.cu")
text = source.read_text()
if "#include <cfloat>" not in text:
    source.write_text("#include <cfloat>\n" + text)
%pip install plyfile
%pip install ./submodules/diff-gaussian-rasterization
%pip install ./submodules/simple-knn
import diff_gaussian_rasterization
import simple_knn

## 3. Supply the dataset and visual checkpoint
Obtain the archive through the upstream project and place `RF-3DGS_dataset.zip` in your own Google Drive. This repository does not include the archive or trained weights.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
archive = Path("/content/drive/MyDrive/RF-3DGS_dataset.zip")
assert archive.is_file(), "Add the upstream dataset archive to your Drive first."
!unzip -q -n "{archive}" -d /content/dataset
DATA = Path("/content/dataset/RF-3DGS_dataset")
SCENE = DATA / "training-rf-spectrum" / "3dgs_MPC_100"
CHECKPOINT = DATA / "blender_visual_trained" / "chkpnt30000.pth"
MODEL = Path("/content/RF-3DGS/output/rf3dgs_MPC")
assert SCENE.is_dir() and CHECKPOINT.is_file(), "Check the archive layout."

## 4. Checkpoint compatibility
The original notebook explicitly set `weights_only=False` for its trusted upstream initialization checkpoint. Only load checkpoints from sources you trust: this deserialization mode can execute pickle code. The patch is limited to the upstream training script in this runtime.

In [ ]:
training_script = Path("train.py")
text = training_script.read_text()
old = "torch.load(checkpoint)"
new = "torch.load(checkpoint, weights_only=False)"
if old in text:
    training_script.write_text(text.replace(old, new))
elif new not in text:
    raise RuntimeError("Upstream checkpoint loading changed; inspect train.py before continuing.")

## 5. Train from the visual checkpoint
Resume from iteration 30,000 and continue through iteration 40,000. The evaluation/save schedule matches the original completed experiment.

In [ ]:
!python train.py -s {SCENE} -m {MODEL} --iterations 40000 --start_checkpoint {CHECKPOINT} --eval --test_iterations 35000 40000 --save_iterations 40000

## 6. Render and compare RF views

In [ ]:
!python render.py -s {SCENE} -m {MODEL} --iteration 40000

In [ ]:
from IPython.display import display, Image
render_dir = MODEL / "test" / "ours_40000"
if not (render_dir / "renders").is_dir():
    render_dir = MODEL / "train" / "ours_40000"
print("Displayed split:", render_dir.parent.name)
images = sorted((render_dir / "renders").glob("*.png"))[:3]
assert images, "Check the rendering output before continuing."
for path in images:
    print(path.name, "— ground truth, then reconstruction")
    display(Image(filename=str(render_dir / "gt" / path.name)))
    display(Image(filename=str(path)))

## 7. Save results
Archive the generated model and rendered views to your own Drive. Check storage space before running this cell.

In [ ]:
import shutil
export = Path("/content/drive/MyDrive/gaussian_splatting_results")
export.mkdir(parents=True, exist_ok=True)
result = shutil.make_archive(str(export / "rf3dgs_MPC_results"), "zip", root_dir=MODEL)
print("Saved:", result)

## Interpretation
RF spectrum reconstruction and optical image reconstruction are different tasks. Their PSNR values should not be ranked against each other. See [experiment notes](../docs/EXPERIMENTS.md) for metrics extracted from the original saved logs.